In [8]:
import tensorflow as tf
import librosa
import numpy as np
import joblib
from pydub import AudioSegment
import os
import tempfile
import math

def convert_to_wav(input_path, output_path):
    """
    Convert audio file to WAV format using pydub.
    
    Args:
        input_path (str): Path to input audio file (any format).
        output_path (str): Path to save WAV file.
    
    Returns:
        str: Path to converted WAV file, or None if conversion fails.
    """
    try:
        if not os.path.exists(input_path):
            return None
        if input_path.lower().endswith('.wav'):
            return input_path
        try:
            AudioSegment.ffmpeg
        except Exception as e:
            print(f"Error: ffmpeg not found. Please install ffmpeg and add it to PATH. Details: {e}")
            return None
        audio = AudioSegment.from_file(input_path)
        audio.export(output_path, format="wav")
        return output_path
    except Exception as e:
        print(f"Error converting {input_path} to WAV: {e}")
        return None

def evaluate_audio(audio_file_path, model_path='cnn_lstm_multi_output_model.keras', 
                   scaler_X_path='scaler_X.pkl', scaler_y_path='scaler_y.pkl', 
                   segment_duration=5.0):
    """
    Evaluate an audio file by converting it to WAV, segmenting it, predicting scores for each segment,
    and computing the average of the top 50% scores for Pronunciation and Fluency.
    
    Args:
        audio_file_path (str): Path to the audio file (any format).
        model_path (str): Path to the saved model (.keras file).
        scaler_X_path (str): Path to the saved StandardScaler for features.
        scaler_y_path (str): Path to the saved MinMaxScaler for targets.
        segment_duration (float): Duration of each segment in seconds (default: 5.0).
    
    Returns:
        dict: Dictionary with average 'Pronunciation' and 'Fluency' scores (1 to 10) from the top 50% segments.
    """
    # Load the saved model and scalers
    try:
        model = tf.keras.models.load_model(model_path)
        scaler_X = joblib.load(scaler_X_path)
        scaler_y = joblib.load(scaler_y_path)
    except Exception as e:
        raise ValueError(f"Error loading model or scalers: {str(e)}")
    
    # Create a temporary file for WAV conversion
    temp_wav_file = None
    try:
        # Convert audio to WAV
        temp_wav_file = tempfile.NamedTemporaryFile(suffix='.wav', delete=False).name
        wav_path = convert_to_wav(audio_file_path, temp_wav_file)
        if wav_path is None:
            raise ValueError(f"Failed to convert {audio_file_path} to WAV")
        
        # Load the WAV file
        audio, sr = librosa.load(wav_path, sr=None)  # Use native sampling rate
        
        # Segment the audio
        segment_samples = int(segment_duration * sr)  # Samples per segment
        segments = [audio[i:i + segment_samples] for i in range(0, len(audio), segment_samples)]
        
        pronun_scores = []
        fluency_scores = []
        max_frames = 100
        
        # Process each segment
        for idx, segment in enumerate(segments):
            if len(segment) < sr * 0.1:  # Skip very short segments (<100ms)
                print(f"Skipping segment {idx}: too short ({len(segment)/sr:.2f} seconds)")
                continue
            
            
            # Extract MFCC features
            mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
            
            # Pad or truncate to max_frames
            if mfccs.shape[1] > max_frames:
                mfccs = mfccs[:, :max_frames]
            else:
                mfccs = np.pad(mfccs, ((0, 0), (0, max_frames - mfccs.shape[1])), mode='constant')

            # Transpose to (max_frames, 13)
            mfccs = mfccs.T  # Shape: (100, 13)
            
            # Standardize features
            mfccs = scaler_X.transform(mfccs.reshape(-1, mfccs.shape[-1])).reshape(1, max_frames, 13)

            # Predict
            predictions = model.predict(mfccs, verbose=0)  # List: [pronun, fluency]
                        
            # Extract scalar scores
            pronun_score = float(predictions[0].flatten()[0])
            fluency_score = float(predictions[1].flatten()[0])
            
            # Inverse transform to 1–10 scale
            scores_array = np.array([[pronun_score, fluency_score]])  # Shape: (1, 2)
            scores = scaler_y.inverse_transform(scores_array)
            pronun_scores.append(float(scores[0, 0]))
            fluency_scores.append(float(scores[0, 1]))
        
        if not pronun_scores or not fluency_scores:
            raise ValueError("No valid segments processed. Audio may be too short or corrupted.")
        
        # Select top 50% of scores
        pronun_scores.sort(reverse=True)
        fluency_scores.sort(reverse=True)

        print(pronun_scores)
        print(fluency_scores)
        top_50_percent_index = math.ceil(len(pronun_scores) * 0.85)
        top_pronun_scores = pronun_scores[:top_50_percent_index]
        top_fluency_scores = fluency_scores[:top_50_percent_index]
        
        # Calculate average of top 50%
        avg_pronun = np.mean(top_pronun_scores)
        avg_fluency = np.mean(top_fluency_scores)
        print(f"Average Pronunciation score (top 50%): {avg_pronun:.2f}")
        print(f"Average Fluency score (top 50%): {avg_fluency:.2f}")
        
        return {
            'Pronunciation': round(avg_pronun, 2),
            'Fluency': round(avg_fluency, 2)
        }
    
    except Exception as e:
        raise ValueError(f"Error processing audio file: {str(e)}")
    
    finally:
        # Clean up temporary WAV file
        if temp_wav_file and os.path.exists(temp_wav_file):
            try:
                os.remove(temp_wav_file)
                print(f"Deleted temporary file: {temp_wav_file}")
            except Exception as e:
                print(f"Warning: Could not delete temporary file {temp_wav_file}: {e}")

# Example usage with a sample audio file
if __name__ == "__main__":
    sample_audio = "C:\\Users\\rahul\\Desktop\\AI-Based-Virtual-Interview-Preparation-Assistant\\Ml Models\\Verbal Communication Evaluator\\motivation.mp3"
    try:
        result = evaluate_audio(sample_audio)
        print(f"\nFinal Evaluation Results:")
        print(f"Pronunciation Score: {result['Pronunciation']}")
        print(f"Fluency Score: {result['Fluency']}")
    except ValueError as e:
        print(f"Evaluation failed: {e}")

Error converting C:\Users\rahul\Desktop\AI-Based-Virtual-Interview-Preparation-Assistant\Ml Models\Verbal Communication Evaluator\motivation.mp3 to WAV: [WinError 2] The system cannot find the file specified
Deleted temporary file: C:\Users\rahul\AppData\Local\Temp\tmp2bgv4u6l.wav
Evaluation failed: Error processing audio file: Failed to convert C:\Users\rahul\Desktop\AI-Based-Virtual-Interview-Preparation-Assistant\Ml Models\Verbal Communication Evaluator\motivation.mp3 to WAV


C:\Users\rahul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\rahul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\rahul\App

In [7]:
!pip install numpy==1.26.4

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: C:\Users\rahul\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
